# Worldwide Earthquake Events API - Silver Layer Processing

In [1]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

StatementMeta(, edde3cba-f64e-455b-9dc7-b76f5e64108f, 3, Finished, Available, Finished, False)

In [4]:
from datetime import date, timedelta
start_date=date.today()-timedelta(7)
print(start_date)

StatementMeta(, edde3cba-f64e-455b-9dc7-b76f5e64108f, 6, Finished, Available, Finished, False)

2026-03-23


In [16]:
# df now is a Spark DataFrame containing JSON data
df = spark.read.option("multiline", "true").json(f"Files/{start_date}_earthquake_data.json")
display(df)

StatementMeta(, edde3cba-f64e-455b-9dc7-b76f5e64108f, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 88ded67e-a1fe-4ece-9871-f738442deb24)

In [17]:
# Reshape earthquake data by extracting and renaming key attributes for further analysis.
df = \
df.\
    select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
        )

StatementMeta(, edde3cba-f64e-455b-9dc7-b76f5e64108f, 19, Finished, Available, Finished, False)

In [18]:
# Convert 'time' and 'updated' columns from milliseconds to timestamp format for clearer datetime representation.
df = df.\
    withColumn('time', col('time')/1000).\
    withColumn('updated', col('updated')/1000).\
    withColumn('time', col('time').cast(TimestampType())).\
    withColumn('updated', col('updated').cast(TimestampType()))

StatementMeta(, edde3cba-f64e-455b-9dc7-b76f5e64108f, 20, Finished, Available, Finished, False)

In [19]:
display(df)

StatementMeta(, edde3cba-f64e-455b-9dc7-b76f5e64108f, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3088e295-fd2e-4008-82a5-f15e31fc3701)

In [20]:
# appending the data to the gold table
df.write.mode('append').saveAsTable('earthquake_events_silver')

StatementMeta(, edde3cba-f64e-455b-9dc7-b76f5e64108f, 22, Finished, Available, Finished, False)